# 02 — Controlled Experimental Dataset

Injects deterministic defects and duplicates into the Bronze dataset for repeatable evaluation.

## Publication notes

- This public notebook contains no credentials, tokens, tenant IDs, subscription IDs, or workspace URLs.
- Notebook outputs were removed to avoid publishing source records or environment metadata.
- Configure the catalog, schema, and source data location for your own Azure environment before execution.
- Run notebooks in numerical order.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import uuid

CATALOG = "dbx_niw_analytics"
SCHEMA = "ztlf"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_customer_churn"
EXPERIMENT_TABLE = f"{CATALOG}.{SCHEMA}.bronze_customer_churn_experiment"
DEFECT_LOG_TABLE = f"{CATALOG}.{SCHEMA}.injected_defect_log"

EXPERIMENT_ID = str(uuid.uuid4())
RANDOM_SEED = 42

print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Target table: {EXPERIMENT_TABLE}")

In [ ]:
source_df = spark.table(SOURCE_TABLE)

source_count = source_df.count()

print(f"Source row count: {source_count}")
display(source_df.limit(10))

In [ ]:
business_columns = [
    "RowNumber",
    "CustomerId",
    "Surname",
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
]

base_df = source_df.select(*business_columns)

print(f"Business columns: {len(base_df.columns)}")

In [ ]:
from pyspark.sql.window import Window
indexed_df = (
    base_df
    .withColumn(
        "_experiment_row_number",
        F.row_number().over(
            Window.orderBy(
                F.xxhash64(
                    F.col("CustomerId"),
                    F.lit(RANDOM_SEED)
                )
            )
        )
    )
)

In [ ]:
dirty_df = indexed_df.withColumn(
    "EstimatedSalary",
    F.when(
        F.col("_experiment_row_number").between(1, 300),
        F.lit(None).cast("double")
    ).otherwise(F.col("EstimatedSalary"))
)

In [ ]:
dirty_df = dirty_df.withColumn(
    "Age",
    F.when(
        F.col("_experiment_row_number").between(301, 400),
        F.lit(15)
    )
    .when(
        F.col("_experiment_row_number").between(401, 500),
        F.lit(125)
    )
    .otherwise(F.col("Age"))
)

In [ ]:
dirty_df = dirty_df.withColumn(
    "Balance",
    F.when(
        F.col("_experiment_row_number").between(501, 650),
        -F.abs(F.col("Balance")) - F.lit(1.0)
    ).otherwise(F.col("Balance"))
)

In [ ]:
dirty_df = dirty_df.withColumn(
    "Geography",
    F.when(
        F.col("_experiment_row_number").between(651, 700),
        F.lit("france")
    )
    .when(
        F.col("_experiment_row_number").between(701, 750),
        F.lit(" FRANCE ")
    )
    .when(
        F.col("_experiment_row_number").between(751, 800),
        F.lit("GERMANY")
    )
    .when(
        F.col("_experiment_row_number").between(801, 850),
        F.lit("spain ")
    )
    .when(
        F.col("_experiment_row_number").between(851, 900),
        F.lit("United Kingdom")
    )
    .otherwise(F.col("Geography"))
)

In [ ]:
dirty_df = dirty_df.withColumn(
    "Gender",
    F.when(
        F.col("_experiment_row_number").between(901, 950),
        F.lit("male")
    )
    .when(
        F.col("_experiment_row_number").between(951, 1000),
        F.lit(" FEMALE ")
    )
    .when(
        F.col("_experiment_row_number").between(1001, 1050),
        F.lit("M")
    )
    .when(
        F.col("_experiment_row_number").between(1051, 1100),
        F.lit("Unknown")
    )
    .otherwise(F.col("Gender"))
)

In [ ]:
dirty_df = dirty_df.withColumn(
    "CustomerId",
    F.when(
        F.col("_experiment_row_number").between(1101, 1150),
        F.lit(None).cast(LongType())
    )
    .when(
        F.col("_experiment_row_number").between(1151, 1200),
        F.lit(-1).cast(LongType())
    )
    .otherwise(F.col("CustomerId"))
)

In [ ]:
duplicate_df = (
    dirty_df
    .filter(
        F.col("_experiment_row_number").between(1201, 1700)
    )
    .withColumn("_is_injected_duplicate", F.lit(True))
)

dirty_original_df = dirty_df.withColumn(
    "_is_injected_duplicate",
    F.lit(False)
)

experimental_df = dirty_original_df.unionByName(duplicate_df)

print(f"Original experimental rows: {dirty_original_df.count()}")
print(f"Injected duplicate rows: {duplicate_df.count()}")
print(f"Combined experimental rows: {experimental_df.count()}")

In [ ]:
experimental_df = (
    experimental_df
    .withColumn("_experiment_id", F.lit(EXPERIMENT_ID))
    .withColumn("_experiment_seed", F.lit(RANDOM_SEED))
    .withColumn("_dataset_type", F.lit("CONTROLLED_DIRTY_DATASET"))
    .withColumn("_created_timestamp", F.current_timestamp())
    .withColumn("_processing_layer", F.lit("BRONZE_EXPERIMENT"))
    .withColumn("_record_status", F.lit("UNVALIDATED"))
)

In [ ]:
validation_metrics = experimental_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("_is_injected_duplicate").cast("int")
    ).alias("injected_duplicate_rows"),

    F.sum(
        F.col("EstimatedSalary").isNull().cast("int")
    ).alias("missing_salary_rows"),

    F.sum(
        (
            (F.col("Age") < 18) |
            (F.col("Age") > 100)
        ).cast("int")
    ).alias("invalid_age_rows"),

    F.sum(
        (F.col("Balance") < 0).cast("int")
    ).alias("negative_balance_rows"),

    F.sum(
        (
            ~F.trim(F.lower(F.col("Geography")))
            .isin("france", "germany", "spain")
        ).cast("int")
    ).alias("invalid_geography_rows"),

    F.sum(
        (
            ~F.trim(F.lower(F.col("Gender")))
            .isin("male", "female")
        ).cast("int")
    ).alias("invalid_gender_rows"),

    F.sum(
        (
            F.col("CustomerId").isNull() |
            (F.col("CustomerId") <= 0)
        ).cast("int")
    ).alias("invalid_customer_id_rows")
)

display(validation_metrics)

In [ ]:
display(
    experimental_df.filter(
        F.col("EstimatedSalary").isNull() |
        (F.col("Age") < 18) |
        (F.col("Age") > 100) |
        (F.col("Balance") < 0) |
        F.col("CustomerId").isNull() |
        (F.col("CustomerId") <= 0) |
        F.col("_is_injected_duplicate")
    ).limit(50)
)

In [ ]:
(
    experimental_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EXPERIMENT_TABLE)
)

print(f"Experimental dataset created: {EXPERIMENT_TABLE}")

In [ ]:
defect_log = [
    ("Duplicate Records",500),
    ("Missing Salary",300),
    ("Invalid Age",200),
    ("Negative Balance",150),
    ("Geography Inconsistency",250),
    ("Gender Inconsistency",200),
    ("Malformed Customer ID",100)
]

defect_log_df = (
    spark.createDataFrame(
        defect_log,
        ["defect_type","expected_count"]
    )
    .withColumn("experiment_id",F.lit(EXPERIMENT_ID))
    .withColumn("created_at",F.current_timestamp())
)

display(defect_log_df)

In [ ]:
(
    defect_log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(DEFECT_LOG_TABLE)
)

print("Defect log saved.")

In [ ]:
spark.sql(f"""
SELECT
COUNT(*) TotalRows,
SUM(CASE WHEN _is_injected_duplicate THEN 1 ELSE 0 END) DuplicateRows
FROM {EXPERIMENT_TABLE}
""").show()